# Ontleding van de Variabiliteit in Schadeafwikkeling over de Organisatiehiërarchie van een Verzekeraar met PROC NESTED

## Samenvatting

Een schade- en ongevallenverzekeraar wil weten **waar** de inconsistentie
in schadeafwikkelingen vandaan komt: wordt die veroorzaakt door
verschillen tussen geografische regio's, tussen vestigingen, tussen
individuele schaderegelaars, of gewoon door claim-tot-claim ruis? Dit
notebook bouwt een synthetisch, volledig genest bestand van
autoschadeclaims (Regio > Vestiging > Schaderegelaar > claim) en gebruikt
**PROC NESTED** om een variantie-analyse met toevalseffecten uit te
voeren, waarbij de variantiecomponent van elk niveau in de hiërarchie
wordt geschat en gerapporteerd als percentage van het totaal. Het
resultaat vertelt het management op welke organisatielaag men zich moet
richten om de afwikkeling consistenter te maken.

## Gegevensbronnen

Alle gegevens worden inline gegenereerd door de eerste DATA-stap — geen
externe bestanden, geen netwerk. Het ontwerp is **volledig genest**: elke
vestiging behoort tot precies één regio, elke schaderegelaar tot precies
één vestiging, en elke claim tot precies één schaderegelaar.

| Dataset | Rijen | Variabele | Type | Rol | Omschrijving |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (niveau 1, buitenste) | Geografische regio (5 regio's: NO, ZO, MW, WK, ZW) |
| | | `Branch` | char | CLASS (niveau 2, genest in Region) | Vestigingscode (2 vestigingen per regio) |
| | | `Adjuster` | char | CLASS (niveau 3, genest in Branch) | ID van de schaderegelaar (2 schaderegelaars per vestiging) |
| | | `Settlement` | num | VAR (afhankelijk) | Uitgekeerde schadevergoeding voor de autoclaim, in USD |
| | | `CycleDays` | num | VAR (afhankelijk) | Dagen van eerste melding tot afwikkeling |

Synthetische structuur: 5 regio's x 2 vestigingen x 2 schaderegelaars x 2
claims = 40 observaties. Een regio-toevalseffect, een
vestiging-binnen-regio-toevalseffect, een
schaderegelaar-binnen-vestiging-toevalseffect en een residu op
claimniveau worden additief opgebouwd via `rand('NORMAL', ...)`, zodat de
variantiecomponenten herleidbaar zijn. Het regio-effect wordt getrokken
met de grootste standaardafwijking (2.200), zodat *waar* een claim wordt
afgehandeld de dominante factor is. Seed vastgezet met
`call streaminit(20260531)`. Een compact, volledig gebalanceerd ontwerp
houdt elk niveau van de hiërarchie gevuld met echte vrijheidsgraden.

# Ontleding van de Variabiliteit in Schadeafwikkeling met PROC NESTED

Wanneer een schade- en ongevallenverzekeraar bekijkt hoeveel er wordt
uitgekeerd om autoclaims af te wikkelen, valt vaak een grote spreiding
op. Een deel daarvan is onvermijdelijk (elk ongeval is anders), maar een
deel weerspiegelt **organisatorische** factoren — de ene regio betaalt
ruimer dan de andere, een vestiging is guller, een individuele
schaderegelaar is een uitschieter.

De gegevens hebben een strikt **geneste** (hiërarchische) structuur:

```
Regio  ->  Vestiging (genest in Regio)  ->  Schaderegelaar (genest in Vestiging)  ->  individuele claims
```

Een vestiging behoort tot precies één regio en een schaderegelaar tot
precies één vestiging, dus de factoren zijn genest in plaats van
gekruist. `PROC NESTED` voert voor precies dit ontwerp een variantie-
analyse met toevalseffecten uit en schat op elk niveau een
**variantiecomponent**, uitgedrukt als percentage van het totaal. Die
percentageverdeling beantwoordt de bedrijfsvraag: *op welke laag van de
organisatie moeten we ons richten om schadeafwikkelingen consistenter te
maken?*

## Stap 1 — Genereer een synthetisch, volledig genest claimbestand

We simuleren 5 regio's, elk met 2 vestigingen, elk met 2 schaderegelaars,
elk die 2 claims behandelt (40 rijen in totaal). We bouwen de respons op
uit additieve toevalseffecten, zodat elk niveau echt variantie
bijdraagt:

- een **regio**-effect (grootste spreiding — regio's verschillen het
  meest),
- een **vestiging-binnen-regio**-effect,
- een **schaderegelaar-binnen-vestiging**-effect,
- en een **residu op claimniveau** (de ruis binnen een schaderegelaar).

`call streaminit` legt de seed vast voor reproduceerbaarheid; elk effect
wordt getrokken met `rand('NORMAL', mean, sd)`. Het regio-effect
gebruikt de breedste standaardafwijking, zodat het het grootste aandeel
van de totale variantie zou moeten opeisen. We genereren ook een tweede
respons, `CycleDays`, die dezelfde hiërarchie deelt, zodat we later de
analyse met meerdere responsen kunnen demonstreren.

In [1]:
GEGEVENS claims;
   CALL streaminit(20260531);
   LENGTE Region $2 Branch $6 Adjuster $9;

   /* Effect op regioniveau: regio's verschillen het meest. */
   DOE r = 1 TOT 5;
      Region = SCAN('NO ZO MW WK ZW', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Vestiging genest binnen regio. */
      DOE b = 1 TOT 2;
         Branch = catx('-', Region, put(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Schaderegelaar genest binnen vestiging. */
         DOE a = 1 TOT 2;
            Adjuster = catx('-', Branch, put(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Individuele claims behandeld door deze schaderegelaar. */
            DOE claim = 1 TOT 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               ALS Settlement < 0 DAN Settlement = 0;
               ALS CycleDays  < 1 DAN CycleDays  = 1;
               UITVOER;
            EINDE;
         EINDE;
      EINDE;
   EINDE;

   BEWAREN Region Branch Adjuster Settlement CycleDays;
UITVOEREN;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Stap 2 — Sorteer op de nestingshiërarchie

`PROC NESTED` vereist dat de invoergegevens **gesorteerd zijn op de
CLASS-variabelen in de volgorde waarin ze worden opgegeven** — buitenste
factor eerst. We sorteren op `Region Branch Adjuster`, zodat de
procedure de hiërarchie correct kan doorlopen.

In [2]:
PROCEDURE SORTEREN GEGEVENS=claims;
   VOLGENS Region Branch Adjuster;
UITVOEREN;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Stap 3 — Variantiecomponentenanalyse van het schadebedrag

De kernanalyse. We geven de CLASS-variabelen buitenste-naar-binnenste op
(`Region Branch Adjuster`); de binnenste herhaling — de individuele
claims — wordt automatisch als foutterm behandeld. `VAR Settlement`
noemt de enige respons.

De `LABEL`-instructie levert een vriendelijkere weergavenaam voor de
respons in de uitvoerbanner. De uitvoer produceert de coëfficiënten van
de verwachte gemiddelde kwadraten, de variantie-analysetabel (met een
F-toets van elke variantiecomponent tegen nul), en de schatting van de
**variantiecomponent** op elk niveau samen met het **percentage van het
totaal**.

In [3]:
TITEL 'Variantiecomponenten van Autoclaimschikkingen';
title2 'Regio / Vestiging / Schaderegelaar - Random-Effects ANOVA';

PROCEDURE nested GEGEVENS=claims;
   KLASSE Region Branch Adjuster;
   VARIABELE Settlement;
   label Region='Regio' Branch='Vestiging' Adjuster='Schaderegelaar'
         Settlement = 'Schadevergoeding (USD)';
UITVOEREN;


                                     Variantiecomponenten van Autoclaimschikkingen                                      
                               Regio / Vestiging / Schaderegelaar - Random-Effects ANOVA                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadevergoeding (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regio                  4  62114122.126866          6.71      0.0304  Vestiging  15528530.531717  1651824.844989             54.4057      8.0000
Vestiging              5  11569658.859028          1.89      0.1829  Schaderegelaar  2313931.771806   272682.828942              8.9813      4.0000
Schaderegelaar        10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000


NOTE: Option TITLE changed to Variantiecomponenten van Autoclaimschikkingen.
NOTE: Option TITLE2 changed to Regio / Vestiging / Schaderegelaar - Random-Effects ANOVA.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Stap 4 — Analyseer twee responsen tegelijk

Het management is ook geïnteresseerd in de **doorlooptijd** — hoe lang
het duurt voordat claims worden afgewikkeld. We voegen `CycleDays` toe
aan de `VAR`-lijst. Met meer dan één afhankelijke variabele rapporteert
`PROC NESTED` bovendien een **covariantie-analyse** die laat zien hoe de
twee responsen op elk niveau van de hiërarchie samen variëren
(bijvoorbeeld of de regio's die meer uitkeren ook langzamer afwikkelen).

In [4]:
TITEL 'Variantiecomponenten van Schadevergoeding en Doorlooptijd';
title2 'Twee Responsvariabelen over Dezelfde Geneste Hiërarchie';

PROCEDURE nested GEGEVENS=claims;
   KLASSE Region Branch Adjuster;
   VARIABELE Settlement CycleDays;
   label Region='Regio' Branch='Vestiging' Adjuster='Schaderegelaar'
         Settlement = 'Schadevergoeding (USD)'
         CycleDays  = 'Dagen tot Afwikkeling';
UITVOEREN;


                               Variantiecomponenten van Schadevergoeding en Doorlooptijd                                
                                Twee Responsvariabelen over Dezelfde Geneste Hiërarchie                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadevergoeding (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regio                  4  62114122.126866          6.71      0.0304  Vestiging  15528530.531717  1651824.844989             54.4057      8.0000
Vestiging              5  11569658.859028          1.89      0.1829  Schaderegelaar  2313931.771806   272682.828942              8.9813      4.0000
Schaderegelaar        10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000


NOTE: Option TITLE changed to Variantiecomponenten van Schadevergoeding en Doorlooptijd.
NOTE: Option TITLE2 changed to Twee Responsvariabelen over Dezelfde Geneste Hiërarchie.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Stap 5 — Alleen-variantie-weergave met de AOV-optie

Voor een snel overzicht van de variantiecomponenten over beide responsen
beperkt de optie `AOV` de uitvoer tot de variantie-analysestatistieken
en **onderdrukt de covariantie-analyse**. Dit is het compacte overzicht
dat een portefeuilleanalist zou scannen wanneer alleen de
variantieverdeling per niveau nodig is voor elke respons, niet de
covariantie tussen de responsen.

In [5]:
TITEL 'Alleen Variantiecomponenten (AOV)';

PROCEDURE nested GEGEVENS=claims aov;
   KLASSE Region Branch Adjuster;
   VARIABELE Settlement CycleDays;
   label Region='Regio' Branch='Vestiging' Adjuster='Schaderegelaar'
         Settlement = 'Schadevergoeding (USD)'
         CycleDays  = 'Dagen tot Afwikkeling';
UITVOEREN;

TITEL;


                                           Alleen Variantiecomponenten (AOV)                                            
                                Twee Responsvariabelen over Dezelfde Geneste Hiërarchie                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadevergoeding (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regio                  4  62114122.126866          6.71      0.0304  Vestiging  15528530.531717  1651824.844989             54.4057      8.0000
Vestiging              5  11569658.859028          1.89      0.1829  Schaderegelaar  2313931.771806   272682.828942              8.9813      4.0000
Schaderegelaar        10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000


NOTE: Option TITLE changed to Alleen Variantiecomponenten (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## De resultaten interpreteren

De kolom **Percentage van het Totaal** in de variantie-analysetabel is
het belangrijkste cijfer. Van boven naar beneden gelezen geeft die het
aandeel van de totale variabiliteit in de schadevergoeding dat aan elke
organisatorische laag kan worden toegeschreven. Voor het schadebedrag
levert de run op:

- **Regio — 54,4%** (Pr > F = 0,0304). De gegevens zijn gegenereerd met
  de grootste spreiding op regioniveau, en de analyse herleidt dit:
  meer dan de helft van alle variabiliteit in de schadevergoeding zit
  *tussen* regio's — statistisch significant bewijs dat *waar* een claim
  wordt afgehandeld de dominante factor is.
- **Vestiging (binnen Regio) — 9,0%** (Pr > F = 0,18). Een bescheiden
  extra aandeel wanneer je van regio naar individuele vestiging afdaalt;
  hier niet significant.
- **Schaderegelaar (binnen Vestiging) — 3,7%** (Pr > F = 0,33). Weinig
  variatie is in dit bestand herleidbaar tot de individuele
  schaderegelaar.
- **Residu — 32,9%.** De resterende ruis van claim tot claim binnen een
  schaderegelaar; dit is de onvermijdelijke variatie die geen
  organisatorische hefboom kan wegnemen.

Elk niveau draagt ook een **F-toets (Pr > F)** van de nulhypothese dat
zijn variantiecomponent nul is. De kleine p-waarde voor Regio (0,0304)
is statistisch bewijs voor echte systematische verschillen tussen
regio's, geen toeval van de steekproef.

De respons voor doorlooptijd vertelt een parallel verhaal: **Regio is
goed voor 45,8%** van de variatie in dagen-tot-afwikkeling (Pr > F =
0,0448, opnieuw significant), waarbij Vestiging en Schaderegelaar
eencijferige aandelen bijdragen en het residu 40,1% draagt. Zowel *hoeveel*
een verzekeraar uitkeert als *hoe lang* het duurt, worden dus in de
eerste plaats bepaald door regio.

De run met twee responsen voegt een **covariantie-analyse** toe. Hier
stuurt het niveau Regio de kruisproducten, en de totale
**correlatiecoëfficiënt is -0,36** — een negatief verband: de regio's
die grotere schadevergoedingen uitkeren, wikkelen doorgaans juist
*sneller* af, niet langzamer. Dat is een nuttige, niet-voor-de-hand-
liggende bevinding — de dure regio's zijn niet de trage regio's.

**Zakelijke conclusie:** omdat Regio de percentageverdeling voor beide
responsen domineert, moet de verzekeraar eerst de
afwikkelingsrichtlijnen en bevoegdheidslimieten *over regio's heen*
standaardiseren — dat is waar de grootste, statistisch significante
inconsistentie zit — voordat wordt geïnvesteerd in interventies op
vestigings- of schaderegelaarniveau. De negatieve correlatie tussen
schadevergoeding en doorlooptijd betekent dat er geen enkele regio is
die zowel het duurst als het traagst is, dus de twee problemen kunnen
worden aangepakt met aparte, regiogerichte programma's. PROC NESTED
verandert een vaag gevoel van "onze afwikkelingen zijn inconsistent" in
een concrete, laag-voor-laag toewijzing van waar die inconsistentie
vandaan komt.